# UART 波特率配置与诊断 · 04

支持 **9600、19200、38400、57600、115200、230400 baud**；帧格式固定 **8N1、3.3V TTL**。默认硬件档位为115200，界面读取硬件返回的档位，不会仅修改显示文字。

## 启动前

1. 停止 Windows 发送；保存03的记录，然后在 **03笔记本选择 File → Close and Halt**，结束旧内核。
2. 在板上 Jupyter 根目录创建 **`uart_baud`** 文件夹，将 `04_uart_baud_panel.ipynb`、`uart_diag_baud.py` 和新生成的配套 `uart_diag_baud.bit`、`uart_diag_baud.hwh` 放在这里。
3. 打开本笔记本，选择 **Cell → Run All**。本页只有两个Python代码格：先定义面板，再启动。下方Windows命令只是说明，不会被Run All执行。

启动/重跑启动格会重新加载硬件并回到默认115200。请先结束发送、读取并导出旧数据。03/04使用共同内核锁；02及其他脚本不使用此锁，仍需手动停止。

## 每轮怎么做

1. 停止发送并读空上一轮 FIFO。需要换档时，选波特率后点 **应用波特率**，等面板显示“硬件回读确认”。成功换档会保存已读记录、复位接收器并开始新记录。
2. 同一档位重复测试时，先点 **开始新记录**。此按钮会清空硬件FIFO；先读完需要保留的数据。已有软件记录自动导出成功后才清空。
3. 在Windows指定同一 `--baud` 发送短突发，再回面板点 **读取数据**。
4. 点 **导出 CSV**，下载按钮下的CSV和同编号JSON到电脑。板上文件在 `uart_baud/panel_exports`。链接若被浏览器预览，可从Jupyter文件列表勾选文件后下载。

每次读取最多256条，表格显示最近200条，本轮软件保留最多10000条；**硬件FIFO只有1024条，软件容量不等于硬件容量**。当前为手动读取，持续高速发送可能溢出。错误帧的字节无效，显示`--`，CSV留空。硬件访问异常时保留旧记录及原波特率，可导出，读取与换档会停用。

JSON中的 `hardware_baud_setting` 是硬件回读的**标称档位**，`calculated_baud` 是100MHz除以整数分频数的理论值，**不是实测线速**；时间字段是Python主机操作时间，不是FPGA逐字节时间戳。


## 1. 定义面板

先运行此格；它只定义类和按钮逻辑。


In [3]:
import csv, json, fcntl
import importlib.util
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import ipywidgets as widgets
from IPython.display import display


def utc_now():
    return datetime.now(timezone.utc).isoformat()


class UARTBaudPanel:
    LIMIT, PAGE, BATCH = 10000, 200, 256
    BAUD_RATES = (9600, 19200, 38400, 57600, 115200, 230400)

    def __init__(self, rx, base):
        self.rx, self.base, self.records = rx, Path(base), []
        self.failed, self.active, self.overflow = False, True, False
        self.state, self.snapshot_at, self.snapshot_ok = {}, None, False
        self.configuration, self.attempted_baud = None, None
        self.started_at, self.links = utc_now(), ""
        self.read_button = widgets.Button(description="读取数据", button_style="primary")
        self.new_button = widgets.Button(description="开始新记录")
        self.export_button = widgets.Button(description="导出 CSV")
        self.apply_button = widgets.Button(description="应用波特率")
        self.baud_select = widgets.Dropdown(options=self.BAUD_RATES, value=115200,
                                           description="选择波特率：")
        self.summary, self.table, self.message = widgets.HTML(), widgets.HTML(), widgets.HTML()
        self.bindings = [(self.read_button, self.on_read), (self.new_button, self.on_new),
                         (self.export_button, self.on_export), (self.apply_button, self.on_apply)]
        for button, callback in self.bindings:
            button.on_click(callback)
        self.ui = widgets.VBox([widgets.HTML(value="<b>换波特率前停止 Windows 发送并读空 FIFO；应用成功会开始新一轮记录。</b>"),
                                widgets.HBox([self.baud_select, self.apply_button]),
                                widgets.HBox([self.read_button, self.new_button, self.export_button]),
                                self.summary, self.message, self.table])
        self.new_session(self.rx.status())
        self.baud_select.value = self.state["baud"]
        self.render("硬件已加载；可选择并应用波特率。Windows 发送端必须使用同一档位。")

    def new_session(self, state, requested=None):
        self.records, self.state = [], state
        self.overflow = bool(self.rx.overflow_seen or state["overflow"])
        self.snapshot_at, self.snapshot_ok, self.started_at = utc_now(), True, utc_now()
        self.attempted_baud = None
        self.configuration = {"requested_baud": state["baud"] if requested is None else requested,
                              "hardware_baud_setting": state["baud"],
                              "clocks_per_bit": state["clocks_per_bit"],
                              "calculated_baud": state["calculated_baud"],
                              "calculated_baud_is_measurement": False,
                              "data_bits": 8, "parity": "none", "stop_bits": 1,
                              "io_voltage": "3.3V", "pl_clock_hz": 100000000}

    def snapshot(self):
        state = self.rx.status()
        if state["baud"] != self.configuration["hardware_baud_setting"]:
            raise RuntimeError("硬件波特率被其他程序改变；本轮记录仍保留原配置，当前硬件状态需重新确认")
        self.state = state
        self.overflow = self.overflow or self.rx.overflow_seen or state["overflow"]
        self.snapshot_at, self.snapshot_ok = utc_now(), True

    def fail(self, error):
        self.failed, self.snapshot_ok = True, False
        self.overflow = self.overflow or self.rx.overflow_seen
        self.render("硬件访问异常：{}。已停用读取/清空/换档；可导出已有记录，停止发送后重跑启动格。".format(error))

    def checks(self):
        safe = self.snapshot_ok and not self.overflow and self.state.get("count") == 0
        ramp = (len(self.records) == 64 and
                all(e["data_valid"] and not e["framing_error"] and e["data"] == i
                    for i, e in enumerate(self.records)))
        recovery = ([(e["hex"], bool(e["framing_error"]), bool(e["data_valid"]))
                     for e in self.records] == [("", True, False), ("55", False, True), ("AA", False, True)])
        return {"ramp_64_match": bool(safe and ramp),
                "break_recovery_match": bool(safe and recovery)}

    def render(self, message=""):
        valid = sum(bool(e["data_valid"]) for e in self.records)
        errors = sum(bool(e["framing_error"]) for e in self.records)
        remaining = str(self.state.get("count", "未知"))
        current = str(self.configuration["hardware_baud_setting"])
        if not self.snapshot_ok:
            remaining += "（最后已知值）"
            current += "（本轮记录配置；当前硬件未知）"
        checks = self.checks()
        check_text = ""
        if len(self.records) == 64:
            check_text = "　64 字节 00…3F：" + ("通过" if checks["ramp_64_match"] else "未通过")
        elif len(self.records) == 3:
            check_text = "　BREAK + 55 AA：" + ("通过" if checks["break_recovery_match"] else "未通过")
        self.summary.value = (
            "<b>硬件档位：{} baud / 8N1 / 3.3V</b><br>本轮事件：{} / {}　有效字节：{}　"
            "停止位错误：{}　FIFO 剩余：{}　观察到溢出：<b>{}</b>{}<br>"
            "<small>仅按钮操作时更新；下表显示最近 {} 条。本轮全部记录可导出。</small>"
        ).format(escape(current), len(self.records), self.LIMIT, valid, errors, escape(remaining),
                 "是（本轮可能丢失事件）" if self.overflow else "否", check_text, self.PAGE)
        rows = []
        for e in self.records[-self.PAGE:]:
            good = bool(e["data_valid"]) and not e["framing_error"]
            char = chr(e["data"]) if good and 32 <= e["data"] <= 126 else ("·" if good else "--")
            values = [str(e["index"]), e["hex"] if good else "--", char,
                      "是" if good else "否", "错误" if e["framing_error"] else "正常"]
            style = ' style="color:#a40000;background:#ffe5e5"' if e["framing_error"] else ""
            rows.append("<tr{}>{}</tr>".format(style, "".join(
                '<td style="padding:3px 12px">{}</td>'.format(escape(v)) for v in values)))
        self.table.value = ("<table><thead><tr><th>序号</th><th>HEX</th><th>ASCII</th>"
                            "<th>数据有效</th><th>停止位</th></tr></thead><tbody>" +
                            "".join(rows) + "</tbody></table>")
        if len(self.records) >= self.LIMIT:
            message += " 已达本轮 10000 条上限，请导出后开始新记录。"
        if message:
            self.message.value = "<p>{}</p>{}".format(escape(message), self.links)
        self.read_button.disabled = not self.active or self.failed or len(self.records) >= self.LIMIT
        for control in (self.new_button, self.apply_button, self.baud_select):
            control.disabled = not self.active or self.failed
        self.export_button.disabled = not self.active

    def save(self):
        if not self.failed:
            try:
                self.snapshot()
            except Exception as error:
                self.fail(error)
        directory = self.base / "panel_exports"
        directory.mkdir(exist_ok=True)
        number = 1
        while True:
            stem = "uart_{}_{:04d}".format(self.configuration["hardware_baud_setting"], number)
            csv_path, meta_path = directory / (stem + ".csv"), directory / (stem + ".json")
            number += 1
            if meta_path.exists():
                continue
            try:
                stream = csv_path.open("x", encoding="utf-8-sig", newline="")
                break
            except FileExistsError:
                continue
        with stream:
            writer = csv.writer(stream)
            writer.writerow(["index", "data_hex", "data_valid", "framing_error",
                             "overflow_seen_at_read", "session_overflow_seen",
                             "hardware_baud_setting", "clocks_per_bit"])
            for e in self.records:
                writer.writerow([e["index"], e["hex"] if e["data_valid"] else "",
                                 int(e["data_valid"]), int(e["framing_error"]),
                                 int(e["overflow"]), int(self.overflow), e["baud"], e["clocks_per_bit"]])
        metadata = {"configuration": dict(self.configuration), "event_count": len(self.records),
                    "valid_byte_count": sum(bool(e["data_valid"]) for e in self.records),
                    "framing_error_count": sum(bool(e["framing_error"]) for e in self.records),
                    "session_overflow_seen": bool(self.overflow),
                    "fifo_remaining": self.state.get("count") if self.snapshot_ok else None,
                    "fifo_remaining_last_known": self.state.get("count"),
                    "fifo_snapshot_at_utc": self.snapshot_at, "hardware_error": self.failed,
                    "attempted_baud_after_this_session": self.attempted_baud,
                    "current_hardware_baud": self.state.get("baud") if self.snapshot_ok else None,
                    "session_started_at_utc": self.started_at, "exported_at_utc": utc_now(),
                    "hardware_timestamps": False,
                    "timestamp_note": "时间仅为 Python 主机操作时间，没有 FPGA 接收时间戳；计算线速不是实测值。",
                    "checks": self.checks(), "csv_file": csv_path.name}
        with meta_path.open("x", encoding="utf-8") as stream:
            json.dump(metadata, stream, ensure_ascii=False, indent=2)
        self.links = " · ".join('<a href="/files/uart_baud/panel_exports/{0}" download>{0}</a>'.format(
            escape(p.name)) for p in (csv_path, meta_path))
        return csv_path, meta_path

    def on_read(self, _):
        if not self.active or self.failed or len(self.records) >= self.LIMIT:
            return
        limit, before = min(self.BATCH, self.LIMIT - len(self.records)), len(self.records)
        try:
            for _ in range(limit):
                batch = self.rx.read_events(max_events=1)
                self.records.extend(batch)
                if not batch:
                    break
            self.snapshot()
        except Exception as error:
            self.fail(error)
            return
        self.render("本次读取 {} 条。{}".format(len(self.records) - before,
                    "FIFO 仍有事件，可继续点击读取。" if self.state["count"] else "FIFO 已读空。"))

    def save_before_reset(self):
        if not (self.records or self.overflow):
            return None
        return self.save()[0].name

    def on_new(self, _):
        if not self.active or self.failed:
            return
        try:
            saved = self.save_before_reset()
        except Exception as error:
            self.render("自动导出失败：{}。未清空记录或 FIFO，请处理后重试。".format(error))
            return
        if self.failed:
            self.render("硬件状态异常；未执行清空，请停止发送后重跑启动格。")
            return
        try:
            state = self.rx.clear()
        except Exception as error:
            self.fail(error)
            return
        requested = self.configuration["requested_baud"]
        self.new_session(state, requested)
        self.render(("上一轮已保存为 {}。".format(saved) if saved else "") +
                    "新记录已开始；现在去 Windows 用 {} baud 发送，再回来读取。".format(state["baud"]))

    def on_apply(self, _):
        if not self.active or self.failed:
            return
        target = int(self.baud_select.value)
        try:
            self.snapshot()
        except Exception as error:
            self.fail(error)
            return
        if self.state["count"]:
            self.render("FIFO 还有 {} 条未读事件。先停止发送并点击读取，读空后再应用波特率。".format(self.state["count"]))
            return
        if target == self.state["baud"]:
            self.render("硬件已经是 {} baud；本轮记录保留。开始另一轮测试请点击开始新记录。".format(target))
            return
        try:
            saved = self.save_before_reset()
        except Exception as error:
            self.render("自动导出失败：{}。没有切换波特率，也没有清空记录。".format(error))
            return
        if self.failed:
            self.render("硬件状态异常，未执行换档；停止发送后重跑启动格。")
            return
        self.attempted_baud = target
        try:
            state = self.rx.set_baud(target)
        except ValueError as error:
            self.attempted_baud = None
            self.render("尚未换档：{}".format(error))
            return
        except Exception as error:
            # Retain old records and old configuration even if the write completed.
            self.fail(error)
            return
        self.new_session(state, target)
        self.render(("上一轮已保存为 {}。".format(saved) if saved else "") +
                    "硬件回读确认 {} baud，新记录已开始；Windows 命令也请指定 --baud {}。".format(target, target))

    def on_export(self, _):
        if not self.active:
            return
        try:
            paths = self.save()
            self.render("已保存全部 {} 条：{} 及同名 JSON。{}".format(len(self.records), paths[0].name,
                        "当前硬件未知，已有记录按原配置保存。" if self.failed else ""))
        except Exception as error:
            self.render("导出失败：{}。已有记录仍在内存中，请处理后重试。".format(error))

    def detach(self):
        self.active = False
        for button, callback in self.bindings:
            button.on_click(callback, remove=True)
            button.disabled = True
        self.baud_select.disabled = True
        self.message.value = "<p>此面板已停用，请使用最新启动格下方的面板。</p>"


## 2. 启动面板

运行过上面的定义格后再运行本格，或直接 Cell → Run All。已关闭旧03内核、停止发送后再启动。


In [4]:
# 单独运行启动格时，先给出可操作提示，避免 Path/类名未定义的错误。
if "UARTBaudPanel" not in globals():
    raise RuntimeError("请先运行上面的【1. 定义面板】代码格，或点击 Cell → Run All 顺序运行两个代码格。")

base = Path.cwd()
for filename in ("uart_diag_baud.py", "uart_diag_baud.bit", "uart_diag_baud.hwh"):
    if not (base / filename).is_file():
        raise FileNotFoundError("当前目录缺少 {}：{}。将这三个配套文件与04笔记本放在板上的 uart_baud 文件夹。".format(filename, base))
hwh_nodes = ET.parse(str(base / "uart_diag_baud.hwh")).iter()
if not any(n.get("INSTANCE") == "axi_gpio_diag" for n in hwh_nodes):
    raise RuntimeError("hwh 中没有 axi_gpio_diag，请检查配套 bit/hwh。")

# 03与04共用这把锁；关闭旧笔记本的内核会释放锁，不要手动删除锁文件。
if globals().get("_uart_panel_lock") is None:
    candidate_lock = open("/tmp/pynq_uart_diag_panel.lock", "a")
    try:
        fcntl.flock(candidate_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except OSError:
        candidate_lock.close()
        raise RuntimeError("另一个03/04面板内核正在使用硬件。先在旧笔记本选择 File → Close and Halt，再重试本格。")
    _uart_panel_lock = candidate_lock

spec = importlib.util.spec_from_file_location("uart_baud_panel_driver", str(base / "uart_diag_baud.py"))
driver = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver)
for panel_name in ("_uart_baud_panel", "_uart_panel"):
    old_panel = globals().get(panel_name)
    if old_panel is not None:
        if old_panel.records or old_panel.overflow:
            saved_paths = old_panel.save()  # 保存失败则中止，不重载硬件。
            print("旧面板记录已保存：", *saved_paths)
        old_panel.detach()

from pynq import Overlay
_uart_baud_overlay = Overlay(str(base / "uart_diag_baud.bit"))
if "axi_gpio_diag" not in _uart_baud_overlay.ip_dict:
    raise RuntimeError("Overlay 中没有 axi_gpio_diag，请核对 bit/hwh。")
_uart_baud_panel = UARTBaudPanel(driver.UARTDiagnosticBaud(_uart_baud_overlay.axi_gpio_diag.mmio), base)
display(_uart_baud_panel.ui)


## Windows 发送与验收（只在电脑命令提示符运行）

工作目录：`C:\Users\cici\Desktop\FPGA\uart_baud_stage2`。CH343端口沿用已确认的 **COM5**。

### 先测9600

面板选择9600并点击应用，显示硬件档位9600后，在Windows执行：

```bat
py pc/send_uart_baud.py --port COM5 --baud 9600 --ramp
```

回面板读取：预期 **64条事件、64有效字节、0停止位错误、FIFO剩余0、无溢出，64字节00…3F检查通过**。导出CSV/JSON。

### 再测115200

停止发送后在面板应用115200，Windows执行：

```bat
py pc/send_uart_baud.py --port COM5 --baud 115200 --ramp
```

同样核对并导出。剩余19200、38400、57600、230400按相同方法测试，Windows参数与面板档位一致。**六档支持是设计目标，板级通过结论以实际读取结果为准。**

### 某个档位的异常与恢复

例如面板已设为115200，先点开始新记录，再在Windows运行：

```bat
py pc/send_uart_baud.py --port COM5 --baud 115200 --break-recovery
```

回面板读取，预期 **3条事件、2有效字节、1停止位错误、FIFO剩余0、无溢出**；第一行红色错误且字节`--`，随后正常`55`和`AA`。20ms BREAK、100ms间隔都是软件请求时间，不是示波器实测值，也不是接收器恢复延迟。

本页只做配置/读取/导出，没有自动遍历板级测试或自动判断整个作品完成。


In [5]:
from io import BytesIO
from zipfile import ZipFile, ZIP_DEFLATED
from base64 import b64encode
from IPython.display import HTML, display

_dl_panel = globals().get("_uart_baud_panel")
if _dl_panel is None:
    raise RuntimeError("请在当前已运行的04笔记本中执行这一格。")

_dl_dir = _dl_panel.base / "panel_exports"
_dl_files = sorted(p for p in _dl_dir.glob("uart_*")
                   if p.is_file() and p.suffix in (".csv", ".json"))
print("记录位置：", _dl_dir)
if not _dl_files:
    raise RuntimeError("这里没有找到 CSV/JSON，请把这格输出发给我。")

_dl_buffer = BytesIO()
with ZipFile(_dl_buffer, "w", ZIP_DEFLATED) as _dl_zip:
    for _dl_file in _dl_files:
        _dl_zip.writestr(_dl_file.name, _dl_file.read_bytes())

_dl_data = b64encode(_dl_buffer.getvalue()).decode("ascii")
print("已读到 {} 个文件，点击下面的新链接：".format(len(_dl_files)))
display(HTML(
    '<a download="uart_baud_records.zip" '
    'href="data:application/zip;base64,{}">'
    '下载全部已保存记录（ZIP）</a>'.format(_dl_data)
))

记录位置： /home/xilinx/jupyter_notebooks/uart_diag/uart_baud/panel_exports
已读到 24 个文件，点击下面的新链接：


In [9]:
from base64 import b64encode
from IPython.display import HTML, display

_nb_file = _uart_baud_panel.base / "04_uart_baud_panel.ipynb"
_nb_data = b64encode(_nb_file.read_bytes()).decode("ascii")

display(HTML(
    '<a download="04_uart_baud_panel.ipynb" '
    'href="data:application/octet-stream;base64,{}">'
    '下载当前04笔记本</a>'.format(_nb_data)
))

In [7]:
p = _uart_baud_panel

assert p.active and not p.failed, "面板状态异常，请保留输出。"
assert not p.records, "本轮已有读取记录，请保留输出，先不要清空。"

before = p.rx.status()
print("读取前：", {k: before[k] for k in ("baud", "count", "overflow")})
assert (before["baud"], before["count"], before["overflow"]) == (115200, 1024, True), "满 FIFO / 溢出状态不符合预期。"

p.on_read(None)  # 等同于点一次“读取数据”，读取256条
after = p.rx.status()

data_ok = len(p.records) == 256 and all(
    e["data_valid"] and not e["framing_error"] and e["data"] == i
    for i, e in enumerate(p.records)
)

print("前256字节是否完整匹配00～FF：", data_ok)
print("读取后：", {k: after[k] for k in ("count", "overflow")})
assert not p.failed and data_ok and after["count"] == 768 and after["overflow"], "读取结果不符合预期，请保留输出。"
print("溢出与前256字节检查通过。")

读取前： {'baud': 115200, 'count': 1024, 'overflow': True}
前256字节是否完整匹配00～FF： True
读取后： {'count': 768, 'overflow': True}
溢出与前256字节检查通过。


本轮经过开发板完全断电再上电，重新加载 UART 硬件与面板。在 115200 baud 下完成两次 64 字节正常接收及一次 BREAK 恢复测试，结果均符合预期。板端系统时间未校准，不作为真实实验日期或恢复时延依据。